In [1]:
import gc
import os
import re
import shutil
import traceback
from collections import defaultdict
from datetime import timedelta

import mne
import numpy as np

In [2]:
# ==========================================================================
# 0. CONSTANTS AND PATHS
# ==========================================================================
# Interval lengths (seconds)
PREICTAL_SEC = 30 * 60               # Preictal window: at most 30 min before onset
INTERICTAL_DIST_SEC = 4 * 60 * 60    # Interictal data must be >= 4 h away from every onset/offset
POSTICTAL_SEC = 60 * 60              # Preictal may not start within 60 min after the previous seizure
CONTINUOUS_TOLERANCE_SEC = 10.0      # Consecutive EDF files <= 10 s apart belong to the same block
EPOCH_LEN_SEC = 10                   # Epoch length

BATCH_SIZE = 1800                    # Epochs per saved .npy batch
AFTER_LAST_SEIZURE = 0               # Key for the interictal windows after the last seizure
                                     # (real seizure_id values start at 1)

# Input / output paths
LOCAL_DATA = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'
SAVE_ROOT = '/kaggle/working/CHB-MIT_Segmentation'

CHANNELS_18 = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ',  'CZ-PZ',
]

In [3]:
# ==========================================================================
# 1. CHANNEL SELECTION AND SEIZURE ANNOTATIONS
# ==========================================================================
def select_channels(raw, channel_list):
    """Normalize channel names and keep only the channels in `channel_list`."""
    if 'T8-P8-0' in raw.ch_names:
        # MNE renames the duplicated 'T8-P8' channel to 'T8-P8-0' / 'T8-P8-1'
        raw.rename_channels({'T8-P8-0': 'T8-P8'})
    raw.rename_channels({ch: ch.strip().replace('EEG ', '').upper() for ch in raw.ch_names})

    existing = [ch for ch in channel_list if ch in raw.ch_names]
    if len(existing) < len(channel_list):
        print(f'[WARN] Only {len(existing)}/{len(channel_list)} channels found: {existing}')
    if existing:
        raw.pick(existing)
    return raw


def load_seizures_from_summary(summary_path, edf_filename):
    """Read (onset, offset) pairs in seconds for one EDF file from the patient's summary.txt."""
    seizures = []
    if not os.path.exists(summary_path):
        return seizures

    with open(summary_path, 'r', encoding='latin-1') as f:
        content = f.read()

    for file_block in content.split('File Name:'):
        if edf_filename.lower() not in file_block.lower():
            continue

        n_match = re.search(r'Number of Seizures in File:\s*(\d+)', file_block)
        if not n_match or int(n_match.group(1)) == 0:
            return seizures

        onsets = re.findall(r'Seizure\s*(?:\d+)?\s*Start Time:\s*(\d+)\s*seconds', file_block)
        offsets = re.findall(r'Seizure\s*(?:\d+)?\s*End Time:\s*(\d+)\s*seconds', file_block)
        seizures = [(float(on), float(off)) for on, off in zip(onsets, offsets)]
        break

    return seizures

In [4]:
# ==========================================================================
# 1.1 QUICK CHECK: NUMBER OF SEIZURES PER PATIENT
# ==========================================================================
seizure_counts = defaultdict(int)

for folder in sorted(os.listdir(LOCAL_DATA)):
    folder_dir = os.path.join(LOCAL_DATA, folder)
    if not os.path.isdir(folder_dir):
        continue

    # Each patient folder has one summary file
    for filename in os.listdir(folder_dir):
        if not filename.endswith('-summary.txt'):
            continue
        with open(os.path.join(folder_dir, filename), 'r') as f:
            matches = re.findall(r'Number of Seizures in File:\s*(\d+)', f.read())
        seizure_counts[folder.upper()] = sum(int(x) for x in matches)
        break

for patient, n in sorted(seizure_counts.items()):
    print(f'{patient}: {n} seizures')

CHB01: 7 seizures
CHB02: 3 seizures
CHB03: 7 seizures
CHB04: 4 seizures
CHB05: 5 seizures
CHB06: 10 seizures
CHB07: 3 seizures
CHB08: 5 seizures
CHB09: 4 seizures
CHB10: 7 seizures
CHB11: 3 seizures
CHB12: 40 seizures
CHB13: 12 seizures
CHB14: 8 seizures
CHB15: 20 seizures
CHB16: 10 seizures
CHB17: 3 seizures
CHB18: 6 seizures
CHB19: 3 seizures
CHB20: 8 seizures
CHB21: 4 seizures
CHB22: 3 seizures
CHB23: 7 seizures
CHB24: 16 seizures


In [5]:
# ==========================================================================
# 2. BUILD THE RECORDING TIMELINE OF ONE PATIENT
# ==========================================================================
def _discover_edf_files(subject_dir):
    return sorted(f for f in os.listdir(subject_dir) if f.lower().endswith('.edf'))


def _read_file_infos(subject_dir, subject, edf_files):
    """Read the header of every EDF file and attach its seizure annotations."""
    summary_path = os.path.join(subject_dir, f'{subject}-summary.txt')
    file_infos = []
    for f in edf_files:
        raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
        raw = select_channels(raw, CHANNELS_18)

        meas_date = raw.info['meas_date']
        if meas_date is None:
            print(f'[WARN] {f}: no meas_date -> skipping file')
            continue

        duration = raw.times[-1] + 1.0 / raw.info['sfreq']
        file_infos.append({
            'file': f,
            'raw': raw,
            'start': meas_date,
            'duration': duration,
            'end': meas_date + timedelta(seconds=duration),
            'seizures': load_seizures_from_summary(summary_path, f),
        })

    file_infos.sort(key=lambda x: x['start'])
    return file_infos


def _group_into_blocks(file_infos):
    """Group files into blocks of (almost) continuous recording."""
    if not file_infos:
        return []

    blocks = [[file_infos[0]]]
    for prev, cur in zip(file_infos, file_infos[1:]):
        gap = (cur['start'] - prev['end']).total_seconds()
        if 0 <= gap <= CONTINUOUS_TOLERANCE_SEC:
            blocks[-1].append(cur)
        else:
            blocks.append([cur])
    return blocks


def _concat_block(block_files):
    """Load, notch-filter (60 Hz) and concatenate the files of one block."""
    raws = []
    for fi in block_files:
        raw = fi['raw'].copy()
        raw.load_data()
        raw.notch_filter(freqs=60.0)
        raws.append(raw)
    raw_concat = raws[0] if len(raws) == 1 else mne.concatenate_raws(raws)
    return raw_concat, block_files[0]['start']


def build_timeline(subject_dir, subject):
    """Return the patient's blocks, sorted by start time.

    Each block is a dict: raw (concatenated mne Raw), start (absolute datetime),
    duration (seconds) and files (the per-file infos it was built from).
    """
    edf_files = _discover_edf_files(subject_dir)
    file_infos = _read_file_infos(subject_dir, subject, edf_files)

    blocks = []
    for block_files in _group_into_blocks(file_infos):
        raw_concat, block_start = _concat_block(block_files)
        blocks.append({
            'raw': raw_concat,
            'start': block_start,
            'duration': raw_concat.times[-1] + 1.0 / raw_concat.info['sfreq'],
            'files': block_files,
        })

    # Block indices are used everywhere downstream, so they must follow time order
    blocks.sort(key=lambda b: b['start'])
    return blocks

In [6]:
# ==========================================================================
# 3. COLLECT ALL SEIZURES OF A PATIENT
#    seizure_id (1..N, in onset order) is used for leave-one-seizure-out (LOSO) evaluation
# ==========================================================================
def collect_seizures(blocks):
    """Return all seizures with block-relative and absolute times, numbered from 1."""
    all_seizures = []
    for b_idx, block in enumerate(blocks):
        block_start = block['start']
        offset_in_block = 0.0        # start of the current file inside the block
        for finfo in block['files']:
            for onset, offset in finfo['seizures']:
                all_seizures.append({
                    'block_idx': b_idx,
                    'onset_in_block': offset_in_block + onset,
                    'offset_in_block': offset_in_block + offset,
                    'onset_abs': block_start + timedelta(seconds=offset_in_block + onset),
                    'offset_abs': block_start + timedelta(seconds=offset_in_block + offset),
                })
            offset_in_block += finfo['duration']

    all_seizures.sort(key=lambda s: s['onset_abs'])
    for i, s in enumerate(all_seizures):
        s['seizure_id'] = i + 1
    return all_seizures

In [7]:
# ==========================================================================
# 4. PREICTAL WINDOWS
# ==========================================================================
def is_preictal_valid(seizure, prev):
    """A seizure gets a preictal window only if it is >= POSTICTAL_SEC + one epoch after the previous one."""
    if prev is None:
        return True
    gap = (seizure['onset_abs'] - prev['offset_abs']).total_seconds()
    return gap >= POSTICTAL_SEC + EPOCH_LEN_SEC


def get_preictal_window(seizure, block, all_seizures):
    """Return (start, end) in seconds relative to `block`, or None if there is not enough preictal data.

    The window ends at the onset, is at most PREICTAL_SEC long and never starts
    within POSTICTAL_SEC after the previous seizure's offset.
    """
    prev = next((x for x in all_seizures if x['seizure_id'] == seizure['seizure_id'] - 1), None)
    if not is_preictal_valid(seizure, prev):
        return None

    onset = seizure['onset_in_block']
    start = max(onset - PREICTAL_SEC, 0)
    if prev is not None:
        postictal_end_abs = prev['offset_abs'] + timedelta(seconds=POSTICTAL_SEC)
        start = max(start, (postictal_end_abs - block['start']).total_seconds())
    return start, onset

In [8]:
# ==========================================================================
# 5. INTERICTAL WINDOWS
# ==========================================================================
def get_lead_seizures(all_seizures, rec_start, min_gap_sec=INTERICTAL_DIST_SEC):
    """Return (seizures sorted by onset, [(seizure_id, seizure)]) for seizures that start
    at least `min_gap_sec` after the previous seizure's offset (or the recording start)."""
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    leads = []
    for i, s in enumerate(seizures):
        prev_end = rec_start if i == 0 else seizures[i - 1]['offset_abs']
        if (s['onset_abs'] - prev_end).total_seconds() >= min_gap_sec:
            leads.append((s['seizure_id'], s))
    return seizures, leads


def get_interictal_windows(blocks, all_seizures):
    """Return interictal windows as (start_abs, end_abs, block_idx), in absolute time,
    sorted by start_abs.

    Interictal = block time minus the INTERICTAL_DIST_SEC margin around every seizure.
    Windows are cut per block because each block has its own raw signal to crop from.
    """
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    margin = timedelta(seconds=INTERICTAL_DIST_SEC)
    windows = []

    for b_idx, block in enumerate(blocks):
        block_start = block['start']
        block_end = block_start + timedelta(seconds=block['duration'])

        # Excluded intervals around every seizure, clipped to this block
        excluded = []
        for s in seizures:
            e0 = max(s['onset_abs'] - margin, block_start)
            e1 = min(s['offset_abs'] + margin, block_end)
            if e0 < e1:
                excluded.append((e0, e1))

        # Merge overlapping excluded intervals
        excluded.sort()
        merged = []
        for e0, e1 in excluded:
            if merged and e0 <= merged[-1][1]:
                merged[-1] = (merged[-1][0], max(merged[-1][1], e1))
            else:
                merged.append((e0, e1))

        # Interictal = complement of the excluded intervals
        cursor = block_start
        for e0, e1 in merged:
            if cursor < e0:
                windows.append((cursor, e0, b_idx))
            cursor = max(cursor, e1)
        if cursor < block_end:
            windows.append((cursor, block_end, b_idx))

    windows.sort(key=lambda w: w[0])
    return windows


def assign_windows_to_seizures(windows, all_seizures, rec_start):
    """Assign interictal windows to the seizure that follows them.

    Each seizure keeps only the windows closest to its onset: a window whose block directly
    follows the block of the last kept window is appended, otherwise it replaces the kept
    windows (it is later in time, hence closer to the onset).
    Windows after the last seizure are stored under AFTER_LAST_SEIZURE.
    Only lead seizures (see get_lead_seizures) receive windows.

    Returns {seizure_id: [(start_abs, end_abs, block_idx), ...]}.
    """
    seizures, leads = get_lead_seizures(all_seizures, rec_start)
    lead_ids = {sid for sid, _ in leads}
    print(f'{len(leads)} seizures for interictal segmentation')

    assigned = {}
    for window in windows:                      # sorted by start_abs
        _, w_end, b_idx = window
        next_sz = next((s for s in seizures if s['onset_abs'] >= w_end), None)
        if next_sz is None:
            sid = AFTER_LAST_SEIZURE
        else:
            sid = next_sz['seizure_id']
            if sid not in lead_ids:
                continue

        if sid in assigned and b_idx == assigned[sid][-1][2] + 1:
            assigned[sid].append(window)        # directly following block: keep both
        else:
            assigned[sid] = [window]            # otherwise keep only the later window

    return assigned

In [9]:
# ==========================================================================
# 6. TESTING SEGMENTS (CONTINUOUS DATA BETWEEN SEIZURES, FOR FAR/h)
# ==========================================================================
def get_testing_segments(blocks, all_seizures, include_tail=False):
    """Return {name: [(block_idx, w0, w1), ...]} with w0/w1 in seconds relative to the block start.

    name is the seizure_id of the seizure that follows the segment, or 'tail' for the
    data after the last seizure (only if include_tail is True).
    Each list is in time order; two consecutive entries with different block_idx mark a
    discontinuity in the recording.
    """
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    rec_start = min(b['start'] for b in blocks)
    rec_end = max(b['start'] + timedelta(seconds=b['duration']) for b in blocks)

    # (name, start_abs, end_abs): from the previous offset (or recording start) to each onset
    spans = []
    prev_end = rec_start
    for s in seizures:
        if prev_end < s['onset_abs']:
            spans.append((s['seizure_id'], prev_end, s['onset_abs']))
        prev_end = max(prev_end, s['offset_abs'])
    if include_tail and prev_end < rec_end:
        spans.append(('tail', prev_end, rec_end))

    # Split every span by block
    result = {}
    for name, lo, hi in spans:
        segments = []
        for b_idx, block in enumerate(blocks):
            b0 = block['start']
            b1 = b0 + timedelta(seconds=block['duration'])
            o0, o1 = max(lo, b0), min(hi, b1)
            if o0 < o1:
                segments.append((b_idx, (o0 - b0).total_seconds(), (o1 - b0).total_seconds()))
        if segments:
            result[name] = segments
    return result

In [10]:
# ==========================================================================
# 7. EPOCHING AND SAVING
# ==========================================================================
def save_fold_npy(data, name, out_dir):
    """Save `data` as <out_dir>/<name>.npy."""
    os.makedirs(out_dir, exist_ok=True)
    free_gb = shutil.disk_usage(out_dir).free / (1024 ** 3)
    print(f'Free disk: {free_gb:.2f} GB')
    np.save(os.path.join(out_dir, f'{name}.npy'), data)


def crop_to_epochs(raw, t_start, t_end, epoch_len=EPOCH_LEN_SEC):
    """Cut raw[t_start:t_end] into fixed-length epochs. Returns None if no valid epoch remains."""
    t_end = min(t_end, raw.times[-1])
    if t_end - t_start < epoch_len:
        return None

    cropped = raw.copy().crop(tmin=max(t_start, 0), tmax=t_end, include_tmax=False)
    epochs = mne.make_fixed_length_epochs(
        cropped, duration=epoch_len, reject_by_annotation=True, preload=True, verbose=False)
    del cropped
    epochs.drop_bad()

    if len(epochs) == 0:
        del epochs
        gc.collect()
        return None
    return epochs


def save_epochs_in_batches(epochs, out_dir, start_batch_idx=0, batch_size=BATCH_SIZE, tag=''):
    """Save epochs as numbered .npy batches; return the next free batch index."""
    n = len(epochs)
    batch_idx = start_batch_idx
    for i in range(0, n, batch_size):
        print(f'{tag} | Epochs: {n} | Batch: {i}:{min(i + batch_size, n)}')
        batch = epochs[i:i + batch_size].get_data(copy=False)
        save_fold_npy(batch, batch_idx, out_dir)
        batch_idx += 1
        del batch
    return batch_idx

In [11]:
# ==========================================================================
# 8. PROCESSING PIPELINE FOR ONE PATIENT
# ==========================================================================
def extract_preictal(blocks, all_seizures, out_dir):
    """Phase 1: one .npy per seizure with its preictal epochs."""
    total = 0
    for s in all_seizures:
        sid = s['seizure_id']
        block = blocks[s['block_idx']]

        window = get_preictal_window(s, block, all_seizures)
        if window is None:
            print(f'  Seizure {sid:02d}: not enough preictal data (skipped)')
            continue

        epochs = crop_to_epochs(block['raw'], *window)
        if epochs is None:
            print(f'  Seizure {sid:02d}: preictal window holds no valid epoch (skipped)')
            continue

        save_fold_npy(epochs.get_data(copy=False), sid, out_dir)
        total += len(epochs)
        del epochs
        print(f'  Seizure {sid:02d}: preictal OK')

    print(f'Total preictal epochs: {total}')
    return total


def extract_interictal(subject, blocks, all_seizures, out_root):
    """Phase 2: interictal epochs of each seizure, saved in batches under seizure_XX/."""
    rec_start = min(b['start'] for b in blocks)
    windows = get_interictal_windows(blocks, all_seizures)
    windows_by_seizure = assign_windows_to_seizures(windows, all_seizures, rec_start)

    total = 0
    for sz_idx, sz_windows in windows_by_seizure.items():
        out_dir = os.path.join(out_root, f'seizure_{sz_idx:02d}')
        os.makedirs(out_dir, exist_ok=True)
        batch_idx = 0

        for a0, a1, b_idx in sz_windows:        # time order, batch numbering continues across windows
            block = blocks[b_idx]
            # absolute -> seconds relative to the block start
            w0 = (a0 - block['start']).total_seconds()
            w1 = (a1 - block['start']).total_seconds()

            epochs = crop_to_epochs(block['raw'], w0, w1)
            if epochs is not None:
                total += len(epochs)
                batch_idx = save_epochs_in_batches(
                    epochs, out_dir, batch_idx,
                    tag=f'Patient: {subject} | Seizure: {sz_idx} | Block: {b_idx}')
            del epochs

    print(f'Total interictal epochs: {total}')
    return total


def extract_testing(blocks, all_seizures, out_root):
    """Phase 3: all signal between seizures, one folder per seizure and block (block_XX),
    so that every folder is a continuous stretch of data."""
    total = 0
    for name, segments in get_testing_segments(blocks, all_seizures).items():
        tag = f'seizure_{name:02d}' if isinstance(name, int) else str(name)
        print(f'Testing: {tag} | {len(segments)} segments')

        for b_idx, w0, w1 in segments:
            out_dir = os.path.join(out_root, tag, f'block_{b_idx:02d}')
            os.makedirs(out_dir, exist_ok=True)

            epochs = crop_to_epochs(blocks[b_idx]['raw'], w0, w1)
            if epochs is not None:
                total += len(epochs)
                save_epochs_in_batches(epochs, out_dir, tag=f'{tag} | Block {b_idx}')
            del epochs

    print(f'Total testing epochs: {total}')
    return total


def process_subject(subject, phase=(1,2,3)):
    phases = {phase} if isinstance(phase, int) else set(phase)
    subject_dir = os.path.join(LOCAL_DATA, subject)
    save_dir = os.path.join(SAVE_ROOT, subject)
    training_dir = os.path.join(save_dir, 'Training')

    print(f'\n--- PROCESSING: {subject.upper()} ---')
    blocks = build_timeline(subject_dir, subject)
    all_seizures = collect_seizures(blocks)

    if not all_seizures:
        print(f'{subject}: no seizures, skipping')
        return
    print(f'{subject}: {len(all_seizures)} seizures in total (1..{len(all_seizures)})')

    if 1 in phases:
        print('PHASE 1: PREICTAL')
        extract_preictal(blocks, all_seizures, os.path.join(training_dir, 'preictal'))

    if 2 in phases:
        print('PHASE 2: INTERICTAL')
        extract_interictal(subject, blocks, all_seizures, os.path.join(training_dir, 'interictal'))

    if 3 in phases:
        print('PHASE 3: TESTING')
        extract_testing(blocks, all_seizures, os.path.join(save_dir, 'testing'))

In [12]:
# ==========================================================================
# 9. PROCESS ALL PATIENTS
# ==========================================================================
START_FROM = 'chb01'   # first patient to process; None = start from the first one
STOP_AT = 'chb03'

def discover_subjects(local_data, start_from=None, stop_at=None):
    """List patient folders (chbXX) sorted by name, optionally limited to [start_from, stop_at]."""
    pattern = re.compile(r'^chb\d{2}$')
    subjects = sorted(d for d in os.listdir(local_data)
                      if pattern.match(d) and os.path.isdir(os.path.join(local_data, d)))
    if start_from is not None:
        subjects = [s for s in subjects if s >= start_from.lower()]
    if stop_at is not None:
        subjects = [s for s in subjects if s <= stop_at.lower()]
    return subjects


if __name__ == '__main__':
    subjects = discover_subjects(LOCAL_DATA, START_FROM, STOP_AT)
    print(f'Found {len(subjects)} patients: {subjects}')
    for subject in subjects:
        try:
            process_subject(subject, phase=(1,2))
        except Exception as e:
            print(f'[ERROR] {subject}: {e}')
            traceback.print_exc()

Found 3 patients: ['chb01', 'chb02', 'chb03']

--- PROCESSING: CHB01 ---


/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/293738

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/293738

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50

...

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50

/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/293738

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
/tmp/ipykernel_16/2937387649.py:13: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)

Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50